# Qwen3-VL Zero-Shot Pipeline  ·  RTFM Scores → Qwen → GPT-4o Judge

**What this notebook does:**
1. Reads pre-computed RTFM anomaly scores + sampled frames from `rtfm_outputs/` (the per-video `metadata.json` files produced by `run_rtfm_pipeline.py`)
2. Sends each video's frames + scores to the **Qwen3-VL-32B server** running in `serve_qwen_colab.ipynb` (zero-shot: no fine-tuning, just a task prompt)
3. Judges every Qwen explanation against human ground truth using **GPT-4o-as-judge** (4 criteria: Correctness, Specificity, Completeness, Fluency)
4. Prints a **head-to-head comparison table** vs the GPT-4o VLM baseline

### Prerequisites
| Step | What to do |
|------|------------|
| 1 | Start `serve_qwen_colab.ipynb` on a **Colab Pro A100** and copy the ngrok URL |
| 2 | Paste the URL into `QWEN_SERVER_URL` in **Cell 1** below |
| 3 | Add `OPENAI_API_KEY` to Colab Secrets (key icon in left sidebar) |
| 4 | Set `RTFM_OUTPUTS_DIR` to where your `rtfm_outputs/` folder lives (Drive or local) |
| 5 | Run cells top-to-bottom |

### Data flow
```
rtfm_outputs/<video_id>/
  metadata.json          ← RTFM gate score, per-snippet scores, sampled frame list
  snippet_*_frame_*.jpg  ← sampled frames
        │
        ▼  POST /explain (frames + scores)
  Qwen3-VL-32B (Colab Pro)  ──►  explanation
        │
        ▼
  GPT-4o-as-judge  ──►  C / S / Co / F scores vs human GT
        │
  qwen_zero_shot_explanations.json
  qwen_zero_shot_judge_summary.json
```

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Configuration
# ══════════════════════════════════════════════════════════════════════════════

# ── 1a. Qwen server URL ───────────────────────────────────────────────────────
# Paste the Cloudflare Tunnel URL printed by serve_qwen_colab.ipynb Cell 4:
QWEN_SERVER_URL = 'https://bat-scenes-navy-intensive.trycloudflare.com'

# ── 1b. Paths ─────────────────────────────────────────────────────────────────
# Option A — running on Colab with Drive mounted:
# RTFM_OUTPUTS_DIR  = '/content/drive/MyDrive/rtfm_pipeline_outputs'
# ANNOTATIONS_PATH  = '/content/drive/MyDrive/annotations.json'

# Option B — running locally (Jupyter):
RTFM_OUTPUTS_DIR  = '/Users/niazahmad/Desktop/Research/explainable-video-anomaly-detection/rtfm/rtfm_pipeline_outputs'
ANNOTATIONS_PATH  = '/Users/niazahmad/Desktop/Research/explainable-video-anomaly-detection/data/SHANGHAI/anomalous_videos/annotations.json'

# Option B (alternative) — relative paths from notebook:
# import os; _here = os.path.dirname(os.path.abspath('__file__'))
# RTFM_OUTPUTS_DIR = os.path.join(_here, '..', 'rtfm_outputs')
# ANNOTATIONS_PATH = os.path.join(_here, '..', '..', 'data', 'SHANGHAI',
#                                  'anomalous_videos', 'annotations.json')

# ── 1c. OpenAI API key (for GPT-4o judge) ────────────────────────────────────
import os
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
    except Exception:
        pass
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

# ── 1d. Tuning knobs ─────────────────────────────────────────────────────────
REQUEST_TIMEOUT  = 180    # seconds to wait per video (32B can be slow on long prompts)
SLEEP_BETWEEN    = 0.5    # seconds between OpenAI judge calls
MAX_NEW_TOKENS   = 300    # max tokens for Qwen generation

# ── Validate ─────────────────────────────────────────────────────────────────
assert QWEN_SERVER_URL, 'Set QWEN_SERVER_URL to the URL from serve_qwen_colab.ipynb'
assert OPENAI_API_KEY,  'Set OPENAI_API_KEY (Colab Secrets or env var)'

QWEN_SERVER_URL = QWEN_SERVER_URL.rstrip('/')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print(f'Qwen server  : {QWEN_SERVER_URL}')
print(f'RTFM outputs : {RTFM_OUTPUTS_DIR}')
print(f'Annotations  : {ANNOTATIONS_PATH}')
print(f'OpenAI key   : {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}')

Qwen server  : https://bat-scenes-navy-intensive.trycloudflare.com
RTFM outputs : /Users/niazahmad/Desktop/Research/explainable-video-anomaly-detection/rtfm/rtfm_pipeline_outputs
Annotations  : /Users/niazahmad/Desktop/Research/explainable-video-anomaly-detection/data/SHANGHAI/anomalous_videos/annotations.json
OpenAI key   : sk-proj-...MVQA


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Mount Drive + install packages  (Colab only)
# ══════════════════════════════════════════════════════════════════════════════
# Skip this cell if running locally in Jupyter.

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except ModuleNotFoundError:
    print('Not in Colab — skipping Drive mount.')

import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai', 'requests', 'tqdm', 'numpy',
])
print('Packages ready.')

Not in Colab — skipping Drive mount.
Packages ready.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Health-check the Qwen server
# ══════════════════════════════════════════════════════════════════════════════
import requests

try:
    r = requests.get(QWEN_SERVER_URL + '/health', timeout=15)
    r.raise_for_status()
    info = r.json()
    print(f'Server OK  |  model={info.get("model")}  |  GPU free={info.get("gpu_free_gb")} GB')
except Exception as e:
    raise RuntimeError(
        f'Cannot reach Qwen server at {QWEN_SERVER_URL}: {e}\n'
        'Make sure serve_qwen_colab.ipynb Cell 4 is running and the URL is correct.'
    )

RuntimeError: Cannot reach Qwen server at https://bat-scenes-navy-intensive.trycloudflare.com: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycloudflare.com/health
Make sure serve_qwen_colab.ipynb Cell 4 is running and the URL is correct.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3b — Simple /explain response test (optional but recommended)
# ══════════════════════════════════════════════════════════════════════════════
# Sends one frame + short prompt to verify the server responds to POST /explain.
# If this fails with 524/530, the tunnel or Colab is timing out; fix server/tunnel first.
import base64
from pathlib import Path
import requests

def _simple_explain_test():
    out_dir = Path(RTFM_OUTPUTS_DIR).resolve()
    # Find first available frame in any video folder
    frame_path = None
    for vfolder in sorted(out_dir.iterdir()):
        if not vfolder.is_dir() or vfolder.name.startswith('.'):
            continue
        for f in sorted(vfolder.iterdir()):
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                frame_path = f
                break
        if frame_path is not None:
            break
    if not frame_path or not frame_path.exists():
        print('Simple /explain test SKIP: no frame found in RTFM_OUTPUTS_DIR')
        return
    with open(frame_path, 'rb') as fh:
        b64 = base64.b64encode(fh.read()).decode('utf-8')
    body = {
        'system_prompt': 'You are a concise anomaly explainer. Reply in one short sentence.',
        'intro_text': 'One sample frame from an anomalous segment (score 0.5). What do you see?',
        'frames': [{'label': 'Frame (score 0.5):', 'b64': b64}],
        'max_new_tokens': 50,
    }
    try:
        r = requests.post(QWEN_SERVER_URL + '/explain', json=body, timeout=90)
        r.raise_for_status()
        expl = r.json().get('explanation', '')[:120]
        print(f'Simple /explain OK  |  response: "{expl}..."')
    except Exception as e:
        print(f'Simple /explain FAIL  |  {e}')
        print('  → Check serve_qwen_colab.ipynb is running and tunnel URL is correct.')
        print('  → 524/530 often mean Cloudflare tunnel timeout; try again or reduce load.')

_simple_explain_test()

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load RTFM outputs (metadata.json) + annotations
# ══════════════════════════════════════════════════════════════════════════════
# We pick each folder inside rtfm_pipeline_outputs (e.g. .../01_0015/). All scores
# are taken from that folder's metadata.json only: gate_score, segment_scores,
# and each extracted_frame's score. Frames + these scores are sent to Qwen in Cell 6.
#
# Segment mapping: each frame has snippet_idx (which segment it belongs to). The
# frame's "score" is that segment's score: frame.score == segment_scores[snippet_idx].
import json
from pathlib import Path

# Resolve to absolute path so loading works regardless of notebook cwd
outputs_dir  = Path(RTFM_OUTPUTS_DIR).resolve()
ann_path     = Path(ANNOTATIONS_PATH).resolve()

# ── Load per-video RTFM metadata: each subfolder = one video (frames + scores) ─
video_metas = {}
for video_folder in sorted(outputs_dir.iterdir()):
    if not video_folder.is_dir() or video_folder.name.startswith('.'):
        continue
    meta_file = video_folder / 'metadata.json'
    if not meta_file.exists():
        continue
    with open(meta_file) as f:
        meta = json.load(f)   # gate_score, segment_scores, extracted_frames[].score from here
    vname = meta.get('video_id', video_folder.name)
    # Only include frames that exist in this folder (scores stay from metadata.json)
    valid_frames = []
    for fr in meta.get('extracted_frames', []):
        fname = fr.get('file')
        if not fname:
            continue
        frame_path = video_folder / fname
        if frame_path.exists():
            fr = dict(fr)
            fr['_abs_path'] = str(frame_path)
            valid_frames.append(fr)
    if not valid_frames:
        continue
    meta = dict(meta)
    meta['video_id'] = vname
    meta['extracted_frames'] = valid_frames
    video_metas[vname] = meta

print(f'RTFM metadata loaded : {len(video_metas)} videos')
print(f'Example IDs          : {list(video_metas)[:5]}')

# ── Load human annotations (ground truth) ────────────────────────────────────
annotations = {}
if ann_path.exists():
    with open(ann_path) as f:
        raw = json.load(f)
    annotations = {e['video_id']: e for e in raw if 'video_id' in e}
    print(f'Human annotations    : {len(annotations)} videos')
else:
    print(f'WARNING: annotations.json not found at {ann_path} — judge will use generic GT for all')

# ── Load existing GPT-4o explanations (for comparison later) ─────────────────
gpt4o_explanations = {}
for expl_file in outputs_dir.glob('*/explanation.json'):
    with open(expl_file) as f:
        d = json.load(f)
    gpt4o_explanations[d['video_id']] = d
print(f'GPT-4o explanations  : {len(gpt4o_explanations)} (for baseline comparison)')

# ── Load existing GPT-4o judge results (for comparison later) ─────────────────
gpt4o_judge_results = []
for jf in sorted(outputs_dir.glob('*/judge.json')):
    with open(jf) as f:
        gpt4o_judge_results.append(json.load(f))
print(f'GPT-4o judge results : {len(gpt4o_judge_results)}')

RTFM metadata loaded : 46 videos
Example IDs          : ['01_0015', '01_0025', '01_0027', '01_0028', '01_0051']
Human annotations    : 44 videos
GPT-4o explanations  : 46 (for baseline comparison)
GPT-4o judge results : 46


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Prompt templates + helper functions
# ══════════════════════════════════════════════════════════════════════════════
import base64, time
import requests
from openai import OpenAI

# ── Zero-shot system prompt (identical to GPT-4o pipeline for fair comparison) ─
SYSTEM_PROMPT = """\
You are a surveillance video anomaly analyst. You will be shown a set of \
frames sampled from a surveillance video that has been flagged as anomalous \
by a weakly-supervised anomaly detection model (RTFM).

The frames are ordered temporally. Each frame comes from a specific temporal \
snippet of the video, and you are given the anomaly score for that snippet \
(0 = normal, 1 = highly anomalous).

The frames were specifically selected from the anomalous portions of the \
video — they represent the onset, peak, and resolution of the detected anomaly.

Your task: Based on ALL the frames and their anomaly scores together, provide \
a single concise explanation (2-3 sentences) of what anomalous activity is \
happening. Focus on:
- WHAT is happening (the specific anomalous activity)
- WHO/WHAT is involved (people, vehicles, objects — describe appearance)
- WHEN in the sequence it starts and ends
- WHY it is anomalous (how it deviates from normal pedestrian behaviour)

Respond with ONLY a JSON object in this exact format:
{\"explanation\": \"...\"}"""

# ── GPT-4o-as-judge prompt ────────────────────────────────────────────────────
JUDGE_PROMPT = """\
You are an impartial judge evaluating the quality of an AI-generated \
explanation of an anomalous event in a surveillance video.

You will be given:
1. A HUMAN ground-truth explanation written by someone who watched the full video.
2. An AI-GENERATED explanation produced by a vision-language model that only \
saw a few sampled frames guided by anomaly scores from a weakly-supervised \
anomaly detection model (RTFM).

Score the AI explanation on these 4 criteria (each 1-5):
- correctness  : Does the AI identify the same anomaly as the human?
  (1 = completely wrong, 3 = right category but wrong details, 5 = exact match)
- specificity  : Does the AI mention specific details (objects, people, actions, location)?
  (1 = very vague, 3 = general activity described, 5 = rich detail matching human level)
- completeness : Does the AI capture all aspects the human mentioned?
  (1 = misses everything, 3 = main point captured but misses secondary details, 5 = covers all)
- fluency      : Is the explanation well-written, clear, and natural?
  (1 = incoherent, 3 = understandable but awkward, 5 = natural and clear)

Also provide a brief justification (1-2 sentences).

Respond with ONLY a JSON object:
{\"correctness\": 1-5, \"specificity\": 1-5, \"completeness\": 1-5, \"fluency\": 1-5, \"justification\": \"...\"}"""

# Generic ground truth for normal videos that were false-positively flagged by RTFM
NORMAL_FP_GT = 'There is nothing anomalous in this video. All pedestrians are walking normally.'


# ── Build request body for POST /explain ─────────────────────────────────────
# meta comes from rtfm_pipeline_outputs/<video_id>/metadata.json; all scores
# (gate_score, per-frame score) are from that file only.
def build_explain_request(meta: dict) -> dict:
    """Construct the JSON body to POST to the Qwen server."""
    frames = meta['extracted_frames']
    segs   = meta.get('anomalous_segments', [])

    # Segment description (uses 'start_snippet'/'end_snippet' keys from run_rtfm_pipeline)
    def seg_label(s):
        start = s.get('start_snippet', s.get('start', '?'))
        end   = s.get('end_snippet',   s.get('end',   '?'))
        return f'snippets {start}-{end}'

    seg_str   = ', '.join(seg_label(s) for s in segs) if segs else 'unknown'
    score_str = ', '.join(
        f"snippet {f['snippet_idx']}={f['score']:.3f}" for f in frames
    )
    intro = (
        f"This video has {meta['n_segments']} temporal snippets (~16 frames each). "
        f"RTFM flagged these contiguous segments as anomalous: [{seg_str}]. "
        f"Video-level anomaly gate score: {meta['gate_score']:.3f} "
        f"(0=normal, 1=highly anomalous; threshold=0.2). "
        f"Below are {len(frames)} frames sampled from the anomalous segments. "
        f"Per-snippet anomaly scores: [{score_str}]."
    )

    frame_items = []
    for fr in frames:
        img_path = fr.get('_abs_path', '')
        if not Path(img_path).exists():
            print(f'  WARN: frame not found: {img_path}')
            continue
        with open(img_path, 'rb') as fh:
            b64 = base64.b64encode(fh.read()).decode('utf-8')
        frame_items.append({
            'label': (
                f"Frame from snippet {fr['snippet_idx']} "
                f"(frame #{fr['frame_num']}, anomaly score: {fr['score']:.3f}):"
            ),
            'b64': b64,
        })

    return {
        'system_prompt':  SYSTEM_PROMPT,
        'intro_text':     intro,
        'frames':         frame_items,
        'max_new_tokens': MAX_NEW_TOKENS,
    }


# ── Call Qwen server ──────────────────────────────────────────────────────────
# 524 = Cloudflare timeout (Colab took too long); retry a few times.
QWEN_MAX_RETRIES = 3
QWEN_RETRY_DELAY = 30   # seconds between retries

def call_qwen_server(body: dict) -> str:
    """POST to /explain, return explanation string. Retries on 524/502/503."""
    last_err = None
    for attempt in range(1, QWEN_MAX_RETRIES + 1):
        try:
            resp = requests.post(
                QWEN_SERVER_URL + '/explain',
                json=body,
                timeout=REQUEST_TIMEOUT,
            )
            resp.raise_for_status()
            return resp.json().get('explanation', '')
        except requests.exceptions.HTTPError as e:
            last_err = e
            if resp.status_code in (502, 503, 524, 530) and attempt < QWEN_MAX_RETRIES:
                print(f' (retry {attempt}/{QWEN_MAX_RETRIES} in {QWEN_RETRY_DELAY}s)', end=' ', flush=True)
                time.sleep(QWEN_RETRY_DELAY)
            else:
                return f'ERROR: {e}'
        except Exception as e:
            return f'ERROR: {e}'
    return f'ERROR: {last_err}'


# ── Call GPT-4o judge ─────────────────────────────────────────────────────────
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def call_gpt4o_judge(human_expl: str, ai_expl: str) -> dict:
    """Score ai_expl against human_expl. Returns dict with 4 scores + justification."""
    user_msg = (
        f'HUMAN ground-truth explanation:\n"{human_expl}"\n\n'
        f'AI-generated explanation:\n"{ai_expl}"'
    )
    try:
        resp = openai_client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role': 'system', 'content': JUDGE_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            max_tokens=300,
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {
            'correctness': None, 'specificity': None,
            'completeness': None, 'fluency': None,
            'justification': f'ERROR: {e}',
        }


print('System prompt, helpers, and OpenAI client ready.')

System prompt, helpers, and OpenAI client ready.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Run Qwen zero-shot inference on all RTFM-flagged videos
# ══════════════════════════════════════════════════════════════════════════════
# For each video:
#   1. Build the multimodal prompt (intro_text with RTFM scores + base64 frames)
#   2. POST to the Qwen server
#   3. Save explanation_qwen_zeroshot.json alongside existing outputs

qwen_explanations = {}   # video_id -> result dict
n_total = len(video_metas)

for i, (vname, meta) in enumerate(sorted(video_metas.items())):
    n_frames  = len(meta['extracted_frames'])
    gate      = meta['gate_score']
    print(f'[{i+1}/{n_total}] {vname}  gate={gate:.3f}  frames={n_frames} ...', end=' ', flush=True)

    body        = build_explain_request(meta)
    n_sent      = len(body['frames'])
    explanation = call_qwen_server(body)

    short = f'"{explanation[:80]}..."' if len(explanation) > 80 else f'"{explanation}"'
    print(f'-> {short}')

    result = {
        'video_id':      vname,
        'model':         'Qwen3-VL-32B-Instruct',
        'gate_score':    gate,
        'n_segments':    meta['n_segments'],
        'anomalous_segments': meta.get('anomalous_segments', []),
        'n_frames_sent': n_sent,
        'explanation':   explanation,
    }
    qwen_explanations[vname] = result

    # Save per-video output
    vid_out = outputs_dir / vname
    vid_out.mkdir(parents=True, exist_ok=True)
    with open(vid_out / 'explanation_qwen_zeroshot.json', 'w') as f:
        json.dump(result, f, indent=2)

# Save batch summary
summary_path = outputs_dir / 'qwen_zeroshot_explanations_summary.json'
with open(summary_path, 'w') as f:
    json.dump(list(qwen_explanations.values()), f, indent=2)

n_ok  = sum(1 for v in qwen_explanations.values() if not v['explanation'].startswith('ERROR'))
n_err = len(qwen_explanations) - n_ok
print(f'\nDone. {n_ok} OK  |  {n_err} errors')
print(f'Saved to: {summary_path}')

[1/46] 01_0015  gate=0.664  frames=4 ... -> "ERROR: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycl..."
[2/46] 01_0025  gate=1.000  frames=8 ... -> "ERROR: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycl..."
[3/46] 01_0027  gate=0.646  frames=8 ... -> "ERROR: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycl..."
[4/46] 01_0028  gate=0.409  frames=7 ... -> "ERROR: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycl..."
[5/46] 01_0051  gate=0.959  frames=8 ... -> "ERROR: 530 Server Error: <none> for url: https://bat-scenes-navy-intensive.trycl..."
[6/46] 01_0052  gate=0.776  frames=7 ... 

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — GPT-4o-as-judge: score every Qwen explanation
# ══════════════════════════════════════════════════════════════════════════════
# 4-digit clip IDs  (e.g. 01_0015)  → truly anomalous → use human annotation as GT
# 3-digit clip IDs  (e.g. 08_023)   → normal FP        → use generic GT

qwen_judge_results = []
n_anom = 0
n_fp   = 0

for i, (vname, expl_data) in enumerate(sorted(qwen_explanations.items())):
    ai_expl = expl_data.get('explanation', '')
    if not ai_expl or ai_expl.startswith('ERROR'):
        print(f'  SKIP {vname}: no valid explanation')
        continue

    # Determine video type by clip ID length
    clip_id = vname.split('_')[1] if '_' in vname else vname
    is_truly_anomalous = len(clip_id) == 4

    if is_truly_anomalous and vname in annotations:
        human_expl = annotations[vname]['explanation']
        gt_start   = annotations[vname].get('anomaly_start_frame')
        gt_end     = annotations[vname].get('anomaly_end_frame')
        video_type = 'anomalous'
        n_anom    += 1
    else:
        human_expl = NORMAL_FP_GT
        gt_start = gt_end = None
        video_type = 'normal_FP'
        n_fp      += 1

    print(f'[{i+1}] {vname}  [{video_type}]')
    ai_short = f'"{ai_expl[:85]}..."' if len(ai_expl) > 85 else f'"{ai_expl}"'
    gt_short = f'"{human_expl[:85]}..."' if len(human_expl) > 85 else f'"{human_expl}"'
    print(f'  Qwen : {ai_short}')
    print(f'  GT   : {gt_short}')

    scores = call_gpt4o_judge(human_expl, ai_expl)
    print(f'  -> C={scores.get("correctness")} '
          f'S={scores.get("specificity")} '
          f'Co={scores.get("completeness")} '
          f'F={scores.get("fluency")}')
    print()

    result = {
        'video_id':          vname,
        'model':             'Qwen3-VL-32B-Instruct',
        'video_type':        video_type,
        'human_explanation': human_expl,
        'ai_explanation':    ai_expl,
        'gate_score':        expl_data.get('gate_score'),
        'gt_anomaly_start':  gt_start,
        'gt_anomaly_end':    gt_end,
        'scores':            scores,
    }
    qwen_judge_results.append(result)

    # Save per-video judge output
    vid_out = outputs_dir / vname
    with open(vid_out / 'judge_qwen_zeroshot.json', 'w') as f:
        json.dump(result, f, indent=2)

    time.sleep(SLEEP_BETWEEN)

# Save batch judge summary
judge_summary_path = outputs_dir / 'qwen_zeroshot_judge_summary.json'
with open(judge_summary_path, 'w') as f:
    json.dump(qwen_judge_results, f, indent=2)

print(f'Judged {len(qwen_judge_results)} videos  ({n_anom} anomalous + {n_fp} normal FP)')
print(f'Saved to: {judge_summary_path}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — Results summary + head-to-head vs GPT-4o baseline
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np

METRICS = ['correctness', 'specificity', 'completeness', 'fluency']


def print_score_table(judge_list, label):
    """Print mean/std/min/max for anomalous videos in judge_list."""
    anom = [r for r in judge_list if r.get('video_type') == 'anomalous']
    if not anom:
        print(f'  {label}: no anomalous results')
        return {}

    rows = {
        m: [r['scores'].get(m) for r in anom if r['scores'].get(m) is not None]
        for m in METRICS
    }

    print(f'\n{"─"*62}')
    print(f'  {label}  (anomalous n={len(anom)})')
    print(f'{"─"*62}')
    print(f'  {"Metric":<16s} {"Mean":>6s} {"Std":>6s} {"Min":>5s} {"Max":>5s}')
    print(f'  {"─"*46}')

    overall = []
    means   = {}
    for m in METRICS:
        v = np.array(rows[m])
        overall.extend(rows[m])
        means[m] = float(v.mean())
        print(f'  {m:<16s} {v.mean():>6.2f} {v.std():>6.2f} {v.min():>5.1f} {v.max():>5.1f}')

    ov = np.array(overall)
    print(f'  {"─"*46}')
    print(f'  {"OVERALL":<16s} {ov.mean():>6.2f} {ov.std():>6.2f} {ov.min():>5.1f} {ov.max():>5.1f}')
    return means


# ── Qwen zero-shot scores ─────────────────────────────────────────────────────
qwen_means = print_score_table(qwen_judge_results, 'Qwen3-VL-32B  (Zero-Shot)')

# ── GPT-4o baseline (from existing judge.json files) ─────────────────────────
if gpt4o_judge_results:
    gpt_means = print_score_table(gpt4o_judge_results, 'GPT-4o  (baseline)')
else:
    print('\n  GPT-4o baseline: no judge.json files found in rtfm_outputs/')
    gpt_means = {}

# ── Per-video delta table (Qwen − GPT-4o) ─────────────────────────────────────
if gpt_means and qwen_means:
    qwen_map = {r['video_id']: r for r in qwen_judge_results if r.get('video_type') == 'anomalous'}
    gpt_map  = {r['video_id']: r for r in gpt4o_judge_results if r.get('video_type') == 'anomalous'}
    common   = sorted(set(qwen_map) & set(gpt_map))

    if common:
        print(f'\n{"="*72}')
        print(f'  Per-video delta  (Qwen Zero-Shot − GPT-4o)  on {len(common)} shared anomalous videos')
        print(f'{"="*72}')
        print(f'  {"Video":<14s} {"ΔC":>5s} {"ΔS":>5s} {"ΔCo":>5s} {"ΔF":>5s}  {"Avg Δ":>7s}')
        print(f'  {"─"*55}')

        deltas = []
        for vname in common:
            qs = qwen_map[vname]['scores']
            gs = gpt_map[vname]['scores']
            d  = {m: (qs.get(m) or 0) - (gs.get(m) or 0) for m in METRICS}
            avg_d = np.mean(list(d.values()))
            deltas.append(avg_d)
            print(f'  {vname:<14s} '
                  f'{d["correctness"]:>+5.0f} {d["specificity"]:>+5.0f} '
                  f'{d["completeness"]:>+5.0f} {d["fluency"]:>+5.0f}  '
                  f'{"+" if avg_d>=0 else ""}{avg_d:>6.2f}')

        mean_d = np.mean(deltas)
        print(f'  {"─"*55}')
        print(f'  {"MEAN DELTA":<14s}{"">22s}  {"+" if mean_d>=0 else ""}{mean_d:>6.2f}')
        print(f'{"="*72}')
        print(f'\n  GPT-4o   overall mean : {np.mean(list(gpt_means.values())):.2f}')
        print(f'  Qwen ZS  overall mean : {np.mean(list(qwen_means.values())):.2f}')
        print(f'  Delta                  : {"+" if mean_d>=0 else ""}{mean_d:.2f}')

# ── Normal FP summary ─────────────────────────────────────────────────────────
fp_results = [r for r in qwen_judge_results if r.get('video_type') == 'normal_FP']
if fp_results:
    print(f'\n  Normal FP videos ({len(fp_results)}) — judged against generic GT:')
    for r in fp_results:
        s = r['scores']
        print(f'    {r["video_id"]:<14s}  C={s.get("correctness")} '
              f'S={s.get("specificity")} '
              f'Co={s.get("completeness")} '
              f'F={s.get("fluency")}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 (optional) — Export results as LaTeX table
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np

anom_results = [r for r in qwen_judge_results if r.get('video_type') == 'anomalous']

if not anom_results:
    print('No anomalous judge results to export.')
else:
    lines = [
        r'\begin{table}[h]',
        r'\centering',
        r'\caption{Qwen3-VL-32B Zero-Shot Anomaly Explanation Results (ShanghaiTech)}',
        r'\label{tab:qwen_zeroshot}',
        r'\begin{tabular}{lrrrr}',
        r'\toprule',
        r'Video ID & Correctness & Specificity & Completeness & Fluency \\',
        r'\midrule',
    ]

    for r in sorted(anom_results, key=lambda x: x['video_id']):
        s = r['scores']
        lines.append(
            f"{r['video_id']} & {s.get('correctness','--')} & "
            f"{s.get('specificity','--')} & {s.get('completeness','--')} & "
            f"{s.get('fluency','--')} \\\\"
        )

    lines.append(r'\midrule')
    for m in METRICS:
        vals = [r['scores'].get(m) for r in anom_results if r['scores'].get(m)]
        col_means = {mm: np.mean([r['scores'].get(mm) for r in anom_results
                                   if r['scores'].get(mm)]) for mm in METRICS}
    lines.append(
        f"Mean & {col_means['correctness']:.2f} & {col_means['specificity']:.2f} & "
        f"{col_means['completeness']:.2f} & {col_means['fluency']:.2f} \\\\"
    )
    lines += [
        r'\bottomrule',
        r'\end{tabular}',
        r'\end{table}',
    ]

    latex = '\n'.join(lines)
    print(latex)

    tex_path = outputs_dir / 'qwen_zeroshot_results.tex'
    with open(tex_path, 'w') as f:
        f.write(latex + '\n')
    print(f'\nSaved LaTeX table to: {tex_path}')